In [1]:
!pip install -q langgraph langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 16.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.


In [2]:
import os
from typing import Annotated, TypedDict
from google.colab import userdata
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# ---------------------------------------------------------------------------
# 0. API Key & Model Setup (Using Colab UserData)
# ---------------------------------------------------------------------------
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("Gemini_API_Key")
except Exception as e:
    raise RuntimeError(
        "Please add 'GOOGLE_API_KEY' to your Colab Secrets (Key icon on the left sidebar) "
        "and enable notebook access."
    ) from e

# Target Gemini 3.6 Flash
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

# Helper function to prevent AttributeError when content is returned as a list
def extract_text_safely(content) -> str:
    if isinstance(content, str):
        return content
    elif isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and "text" in item:
                parts.append(item["text"])
            elif isinstance(item, str):
                parts.append(item)
            else:
                parts.append(str(item))
        return "".join(parts)
    return str(content)

# ---------------------------------------------------------------------------
# 1. State Definition
# ---------------------------------------------------------------------------
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    next_node: str

# ---------------------------------------------------------------------------
# 2. Tool Definition
# ---------------------------------------------------------------------------
@tool
def process_refund(user_id: str, amount: float) -> str:
    """Executes a financial refund for a specific user ID."""
    return f"SUCCESS: Refund of ${amount} has been processed for User '{user_id}'."

# ---------------------------------------------------------------------------
# 3. Agent Nodes
# ---------------------------------------------------------------------------

# Node A: Supervisor Router Agent
def supervisor_agent(state: AgentState) -> AgentState:
    system_prompt = (
        "You are a Support Router. Analyze the user prompt:\n"
        "- If it is general technical troubleshooting, respond with 'SUPPORT'.\n"
        "- If it involves financial refunds or account modifications, respond with 'ACTION'.\n"
        "Respond ONLY with 'SUPPORT' or 'ACTION'."
    )

    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = llm.invoke(messages)

    # Safely convert extracted response to plain string before .strip()
    raw_text = extract_text_safely(response.content)
    decision = raw_text.strip().upper()

    if "ACTION" in decision:
        next_step = "account_actions_agent"
    elif "SUPPORT" in decision:
        next_step = "tech_support_agent"
    else:
        next_step = END

    return {"next_node": next_step}

# Node B: Technical Support Agent
def tech_support_agent(state: AgentState) -> AgentState:
    system_prompt = SystemMessage(content="You are a helpful Technical Support Specialist. Provide concise troubleshooting guidance.")
    messages = [system_prompt] + state["messages"]
    response = llm.invoke(messages)

    text_content = extract_text_safely(response.content)
    return {"messages": [AIMessage(content=f"[Tech Support]: {text_content}")], "next_node": END}

# Node C: Account Action Agent
def account_actions_agent(state: AgentState) -> AgentState:
    llm_with_tools = llm.bind_tools([process_refund])
    system_prompt = SystemMessage(content="You are an Account Manager. Use the process_refund tool to issue user refunds when requested.")

    messages = [system_prompt] + state["messages"]
    response = llm_with_tools.invoke(messages)

    if response.tool_calls:
        tool_call = response.tool_calls[0]
        tool_output = process_refund.invoke(tool_call["args"])
        final_msg = f"[Account Agent]: Executed tool. Result: {tool_output}"
    else:
        text_content = extract_text_safely(response.content)
        final_msg = f"[Account Agent]: {text_content}"

    return {"messages": [AIMessage(content=final_msg)], "next_node": END}

# ---------------------------------------------------------------------------
# 4. Construct LangGraph Workflow
# ---------------------------------------------------------------------------
workflow = StateGraph(AgentState)

workflow.add_node("supervisor", supervisor_agent)
workflow.add_node("tech_support_agent", tech_support_agent)
workflow.add_node("account_actions_agent", account_actions_agent)

workflow.add_edge(START, "supervisor")

workflow.add_conditional_edges(
    "supervisor",
    lambda state: state["next_node"],
    {
        "tech_support_agent": "tech_support_agent",
        "account_actions_agent": "account_actions_agent",
        END: END
    }
)

workflow.add_edge("tech_support_agent", END)
workflow.add_edge("account_actions_agent", END)

app = workflow.compile()

# ---------------------------------------------------------------------------
# 5. Live Demonstration Function
# ---------------------------------------------------------------------------
def run_demo(user_query: str):
    print(f"\n================ USER QUERY ================\n{user_query}")
    inputs = {"messages": [HumanMessage(content=user_query)]}
    result = app.invoke(inputs)
    print("\n================ SYSTEM RESPONSE ================")
    print(result["messages"][-1].content)

# Run Test Cases
run_demo("My app keeps freezing whenever I try to upload a PNG file. How can I fix this?")
run_demo("I was billed twice by mistake. Please refund $49.99 for my account 'user_9876'.")


================ USER QUERY ================
My app keeps freezing whenever I try to upload a PNG file. How can I fix this?

================ SYSTEM RESPONSE ================
[Tech Support]: To fix the freezing issue when uploading a PNG file, try these troubleshooting steps in order:

1. **Test a Different PNG File:** Try uploading a different PNG file to see if the issue is specific to one corrupted or improperly formatted file.
2. **Check File Size:** PNG files can be very large. Compress the image using a free online tool (like TinyPNG) before uploading to reduce memory load.
3. **Clear App Cache & Force Stop:** 
   * **Mobile:** Go to your device **Settings > Apps > [App Name] > Storage**, then tap **Clear Cache** and **Force Stop**.
   * **Desktop/Web:** Clear your browser's cache or restart the desktop app.
4. **Update the App:** Ensure you are running the latest version of the app from your app store or website, as this may be a known bug fixed in a recent patch.
5. **Convert 